In [1]:
import numpy as np
import pandas as pd

# Task 1

In [2]:
np.random.seed(42)
n = 3000

start = pd.Timestamp('2025-01-01 00:00')
end = pd.Timestamp('2025-06-30 23:59')
total_minutes = int((end - start).total_seconds() / 60)

minutes = np.random.randint(0, total_minutes + 1, n)
times = start + pd.to_timedelta(minutes, unit='m')

upi = pd.DataFrame({
    'txn_id': ['T' + str(i).zfill(4) for i in range(1, n + 1)],
    'txn_time': times.strftime('%d/%m/%Y %H:%M'),
    'amount_inr': np.random.randint(10, 5001, n),
    'category': np.random.choice(['Grocery', 'Fuel', 'Recharge', 'Food', 'Bills'], n)
})
upi.to_csv('upi_raw.csv', index=False)

upi = pd.read_csv('upi_raw.csv')
upi['txn_time'] = pd.to_datetime(upi['txn_time'], format='%d/%m/%Y %H:%M')
print(upi.dtypes)

txn_id                   str
txn_time      datetime64[us]
amount_inr             int64
category                 str
dtype: object


# Task 2

In [3]:
upi['year'] = upi['txn_time'].dt.year
upi['month'] = upi['txn_time'].dt.month
upi['day'] = upi['txn_time'].dt.day
upi['weekday'] = upi['txn_time'].dt.day_name()
upi['quarter'] = upi['txn_time'].dt.quarter
upi['hour'] = upi['txn_time'].dt.hour

upi['is_weekend'] = upi['txn_time'].dt.dayofweek >= 5
upi['is_month_end'] = upi['txn_time'].dt.is_month_end

upi['time_slot'] = 'Night'
upi.loc[(upi['hour'] >= 5) & (upi['hour'] <= 11), 'time_slot'] = 'Morning'
upi.loc[(upi['hour'] >= 12) & (upi['hour'] <= 16), 'time_slot'] = 'Afternoon'
upi.loc[(upi['hour'] >= 17) & (upi['hour'] <= 21), 'time_slot'] = 'Evening'

print(upi['time_slot'].value_counts())

time_slot
Night        875
Morning      869
Afternoon    635
Evening      621
Name: count, dtype: int64


# Task 3

In [4]:
indexed = upi.set_index('txn_time').sort_index()

daily_total = indexed['amount_inr'].resample('D').sum()
weekly_total = indexed['amount_inr'].resample('W').sum()
monthly_total = indexed['amount_inr'].resample('ME').sum()

print(daily_total.sum(), weekly_total.sum(), monthly_total.sum())
assert daily_total.sum() == weekly_total.sum() == monthly_total.sum()

daily_total.to_csv('upi_daily.csv')
weekly_total.to_csv('upi_weekly.csv')
monthly_total.to_csv('upi_monthly.csv')

7600666 7600666 7600666


# Task 4

In [5]:
upi['txn_time_ist'] = upi['txn_time'].dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')

print("Before:", upi['txn_time'].iloc[0])
print("After:", upi['txn_time_ist'].iloc[0])
print("Shift:", upi['txn_time_ist'].dt.tz_localize(None).iloc[0] - upi['txn_time'].iloc[0])

busiest_utc = upi['txn_time'].dt.hour.value_counts().idxmax()
busiest_ist = upi['txn_time_ist'].dt.hour.value_counts().idxmax()
print("Busiest hour before conversion:", busiest_utc)
print("Busiest hour after conversion:", busiest_ist)

upi.to_csv('upi_clean.csv', index=False)

Before: 2025-03-26 16:38:00
After: 2025-03-26 22:08:00+05:30
Shift: 0 days 05:30:00
Busiest hour before conversion: 1
Busiest hour after conversion: 20
